# AgentsVille Trip Planner

**Udacity Agentic AI Nanodegree Project**

This notebook implements a multi-agent AI trip planning system that demonstrates:
- Pydantic data models for structured input/output
- Simulated external APIs (weather & activities)
- LLM-powered itinerary generation with Chain-of-Thought reasoning
- Multi-dimensional evaluation of generated plans
- ReACT (Reason-Act-Observe) agent loop for iterative plan revision
- Narrative summary generation

---

## Setup & Imports

In [1]:
import json
import re
import os
from datetime import datetime, timedelta
from typing import Optional

from openai import OpenAI
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

# Env-driven config:
#   OPENAI_API_KEY      — required
#   OPENAI_BASE_URL     — optional, OpenAI-compat endpoint
#   AGENTSVILLE_MODEL   — model name (default gpt-4o-mini)
#   AGENTSVILLE_DISABLE_THINKING  — "1" to disable thinking (needed for glm-4.x)
MODEL_NAME = os.environ.get("AGENTSVILLE_MODEL", "gpt-4o-mini")
DISABLE_THINKING = os.environ.get("AGENTSVILLE_DISABLE_THINKING", "0") == "1"
client = OpenAI()

# Extra body params sent with every request (None → OpenAI SDK drops the arg)
_EXTRA_BODY = {"thinking": {"type": "disabled"}} if DISABLE_THINKING else None


def extract_json(content: str) -> dict:
    """Robust JSON extraction: strips markdown fences, pulls first balanced {...} block."""
    content = content.strip()
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\s*", "", content)
        content = re.sub(r"\s*```$", "", content).strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass
    # Fallback: find first balanced { ... } block
    start = content.find("{")
    if start >= 0:
        depth = 0
        for i, ch in enumerate(content[start:], start):
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return json.loads(content[start:i + 1])
    raise ValueError(f"Could not parse JSON from: {content[:300]}")


print(f"Setup complete. Model={MODEL_NAME} base_url={client.base_url} disable_thinking={DISABLE_THINKING}")


Setup complete. Model=glm-4.6 base_url=https://open.bigmodel.cn/api/paas/v4/ disable_thinking=True


---

## Step 1: VacationInfo Pydantic Model

We define a `VacationInfo` model to capture all the essential details about the traveler's vacation request. Pydantic ensures type safety and validation at runtime.

In [2]:
class VacationInfo(BaseModel):
    """Captures all essential details about a traveler's vacation request."""
    traveler_name: str = Field(description="Full name of the traveler")
    destination_city: str = Field(description="City to visit")
    start_date: str = Field(description="Trip start date in YYYY-MM-DD format")
    end_date: str = Field(description="Trip end date in YYYY-MM-DD format")
    budget: float = Field(description="Total trip budget in USD")
    interests: list[str] = Field(description="List of traveler's interests and preferences")
    special_requirements: str = Field(default="", description="Any special needs or requirements")


# Create our sample vacation request
vacation_info = VacationInfo(
    traveler_name="Alice Johnson",
    destination_city="San Francisco",
    start_date="2025-06-15",
    end_date="2025-06-18",
    budget=2000.0,
    interests=["technology", "food", "outdoor activities", "art"],
    special_requirements="Prefers walking-friendly activities. Vegetarian dining options appreciated."
)

print("VacationInfo Model:")
print(json.dumps(vacation_info.model_dump(), indent=2))

VacationInfo Model:
{
  "traveler_name": "Alice Johnson",
  "destination_city": "San Francisco",
  "start_date": "2025-06-15",
  "end_date": "2025-06-18",
  "budget": 2000.0,
  "interests": [
    "technology",
    "food",
    "outdoor activities",
    "art"
  ],
  "special_requirements": "Prefers walking-friendly activities. Vegetarian dining options appreciated."
}


---

## Step 2: Simulated Weather & Activity APIs

Since we don't have access to live weather or event APIs, we simulate them with realistic data for San Francisco. These functions return structured data that our agents can consume.

In [3]:
# ── Simulated Weather Database ──────────────────────────────────────────────
WEATHER_DATA = {
    "San Francisco": {
        "2025-06-15": {
            "date": "2025-06-15",
            "temperature": "64°F (18°C)",
            "conditions": "Partly cloudy with morning fog",
            "wind_speed": "12 mph"
        },
        "2025-06-16": {
            "date": "2025-06-16",
            "temperature": "68°F (20°C)",
            "conditions": "Sunny and clear",
            "wind_speed": "8 mph"
        },
        "2025-06-17": {
            "date": "2025-06-17",
            "temperature": "61°F (16°C)",
            "conditions": "Overcast with light drizzle",
            "wind_speed": "15 mph"
        },
        "2025-06-18": {
            "date": "2025-06-18",
            "temperature": "66°F (19°C)",
            "conditions": "Mostly sunny, breezy",
            "wind_speed": "10 mph"
        }
    }
}


# ── Simulated Activities Database ──────────────────────────────────────────
ACTIVITIES_DATA = {
    "San Francisco": {
        "2025-06-15": [
            {"name": "Golden Gate Bridge Walk", "type": "outdoor", "cost": 0.0, "duration": "2 hours", "location": "Golden Gate Bridge"},
            {"name": "Exploratorium Museum", "type": "museum", "cost": 30.0, "duration": "3 hours", "location": "Pier 15, Embarcadero"},
            {"name": "Ferry Building Marketplace", "type": "food", "cost": 45.0, "duration": "2 hours", "location": "Ferry Building"},
            {"name": "Alcatraz Island Tour", "type": "historical", "cost": 42.0, "duration": "3 hours", "location": "Alcatraz Island"},
            {"name": "Chinatown Walking Tour", "type": "cultural", "cost": 35.0, "duration": "2 hours", "location": "Chinatown"}
        ],
        "2025-06-16": [
            {"name": "California Academy of Sciences", "type": "museum", "cost": 42.0, "duration": "4 hours", "location": "Golden Gate Park"},
            {"name": "Bike Ride Across Golden Gate", "type": "outdoor", "cost": 35.0, "duration": "3 hours", "location": "Golden Gate Bridge to Sausalito"},
            {"name": "Mission District Food Tour", "type": "food", "cost": 65.0, "duration": "3 hours", "location": "Mission District"},
            {"name": "SFMOMA Visit", "type": "art", "cost": 25.0, "duration": "3 hours", "location": "SoMa District"},
            {"name": "Sunset Kayaking in the Bay", "type": "outdoor", "cost": 55.0, "duration": "2 hours", "location": "Fisherman's Wharf"}
        ],
        "2025-06-17": [
            {"name": "Computer History Museum", "type": "technology", "cost": 22.0, "duration": "3 hours", "location": "Mountain View (day trip)"},
            {"name": "Japanese Tea Garden", "type": "cultural", "cost": 15.0, "duration": "1.5 hours", "location": "Golden Gate Park"},
            {"name": "Ghirardelli Square Chocolate Tour", "type": "food", "cost": 30.0, "duration": "1.5 hours", "location": "Ghirardelli Square"},
            {"name": "Cable Car Ride & Lombard Street", "type": "sightseeing", "cost": 8.0, "duration": "1.5 hours", "location": "Nob Hill to Lombard St"},
            {"name": "SF Public Library & Civic Center", "type": "cultural", "cost": 0.0, "duration": "2 hours", "location": "Civic Center"}
        ],
        "2025-06-18": [
            {"name": "Fisherman's Wharf & Pier 39", "type": "sightseeing", "cost": 20.0, "duration": "2 hours", "location": "Fisherman's Wharf"},
            {"name": "Painted Ladies & Alamo Square", "type": "outdoor", "cost": 0.0, "duration": "1 hour", "location": "Alamo Square"},
            {"name": "Haight-Ashbury Walking Tour", "type": "cultural", "cost": 25.0, "duration": "2 hours", "location": "Haight-Ashbury"},
            {"name": "Tartine Bakery & Coffee", "type": "food", "cost": 25.0, "duration": "1 hour", "location": "Mission District"},
            {"name": "Twin Peaks Sunset Hike", "type": "outdoor", "cost": 0.0, "duration": "2 hours", "location": "Twin Peaks"}
        ]
    }
}


def get_weather_by_date(city: str, date: str) -> dict:
    """Get simulated weather data for a given city and date.
    
    Args:
        city: Name of the city (e.g., "San Francisco")
        date: Date string in YYYY-MM-DD format
        
    Returns:
        Dictionary with date, temperature, conditions, and wind_speed
    """
    if city in WEATHER_DATA and date in WEATHER_DATA[city]:
        return WEATHER_DATA[city][date]
    return {
        "date": date,
        "temperature": "65°F (18°C)",
        "conditions": "Data not available - assuming mild weather",
        "wind_speed": "10 mph"
    }


def get_activities_by_date(city: str, date: str) -> list[dict]:
    """Get simulated available activities for a given city and date.
    
    Args:
        city: Name of the city (e.g., "San Francisco")
        date: Date string in YYYY-MM-DD format
        
    Returns:
        List of activity dictionaries with name, type, cost, duration, location
    """
    if city in ACTIVITIES_DATA and date in ACTIVITIES_DATA[city]:
        return ACTIVITIES_DATA[city][date]
    return [{"name": "General city sightseeing", "type": "sightseeing", "cost": 20.0, "duration": "3 hours", "location": city}]


# ── Demo the APIs ──────────────────────────────────────────────────────────
print("=== Weather for San Francisco on 2025-06-15 ===")
print(json.dumps(get_weather_by_date("San Francisco", "2025-06-15"), indent=2))

print("\n=== Activities for San Francisco on 2025-06-16 ===")
for activity in get_activities_by_date("San Francisco", "2025-06-16"):
    print(f"  - {activity['name']} ({activity['type']}) - ${activity['cost']}")

=== Weather for San Francisco on 2025-06-15 ===
{
  "date": "2025-06-15",
  "temperature": "64\u00b0F (18\u00b0C)",
  "conditions": "Partly cloudy with morning fog",
  "wind_speed": "12 mph"
}

=== Activities for San Francisco on 2025-06-16 ===
  - California Academy of Sciences (museum) - $42.0
  - Bike Ride Across Golden Gate (outdoor) - $35.0
  - Mission District Food Tour (food) - $65.0
  - SFMOMA Visit (art) - $25.0
  - Sunset Kayaking in the Bay (outdoor) - $55.0


---

## Step 3: Pydantic Output Models & Itinerary Agent System Prompt

We define the structured output models (`Activity`, `DayPlan`, `TravelPlan`) and craft a system prompt that instructs the LLM to act as an expert travel planner using Chain-of-Thought reasoning.

In [4]:
# ── Pydantic Output Models ─────────────────────────────────────────────────

class Activity(BaseModel):
    """A single activity within a day plan."""
    name: str = Field(description="Name of the activity")
    type: str = Field(description="Category: outdoor, food, museum, cultural, art, technology, sightseeing, historical")
    cost: float = Field(description="Cost in USD")
    duration: str = Field(description="Expected duration (e.g., '2 hours')")
    location: str = Field(description="Location or venue name")


class DayPlan(BaseModel):
    """Plan for a single day of the trip."""
    date: str = Field(description="Date in YYYY-MM-DD format")
    weather: str = Field(description="Weather summary for the day")
    activities: list[Activity] = Field(description="List of planned activities")
    daily_cost: float = Field(description="Total cost for the day in USD")


class TravelPlan(BaseModel):
    """Complete travel plan for the entire trip."""
    traveler_name: str = Field(description="Name of the traveler")
    destination: str = Field(description="Destination city")
    days: list[DayPlan] = Field(description="Day-by-day plans")
    total_cost: float = Field(description="Total trip cost in USD")
    summary: str = Field(description="Brief summary of the trip plan")


print("Output models defined: Activity, DayPlan, TravelPlan")

Output models defined: Activity, DayPlan, TravelPlan


In [5]:
# ── Itinerary Agent System Prompt ──────────────────────────────────────────

ITINERARY_AGENT_SYSTEM_PROMPT = """You are an expert travel planner with deep knowledge of destinations worldwide. Your role is to create detailed, personalized travel itineraries that match travelers' interests, budgets, and preferences.

## Your Task
Given a traveler's vacation information, weather forecasts, and available activities, create a comprehensive day-by-day travel plan.

## Chain-of-Thought Reasoning
Before generating the plan, think through these steps carefully:
1. **Analyze the traveler's profile**: Consider their interests, budget constraints, and special requirements.
2. **Check weather conditions**: For each day, review the weather and plan activities that are appropriate for those conditions (e.g., indoor activities on rainy days, outdoor activities on sunny days).
3. **Select activities**: Choose activities that align with the traveler's interests from the available options. Ensure variety across days.
4. **Budget management**: Track cumulative costs to stay within the total budget. Distribute spending reasonably across days.
5. **Logistics**: Consider travel time between locations and ensure a realistic schedule.

## Traveler Information
{vacation_info}

## Weather Forecasts
{weather_data}

## Available Activities
{activities_data}

## Output Format (TravelPlan Pydantic Schema)
You MUST respond with ONLY a valid JSON object matching the TravelPlan Pydantic model schema below (no markdown, no extra text):
{{
    "traveler_name": "string",
    "destination": "string",
    "days": [
        {{
            "date": "YYYY-MM-DD",
            "weather": "weather summary string",
            "activities": [
                {{
                    "name": "string",
                    "type": "string",
                    "cost": number,
                    "duration": "string",
                    "location": "string"
                }}
            ],
            "daily_cost": number
        }}
    ],
    "total_cost": number,
    "summary": "Brief summary of the overall trip plan"
}}

IMPORTANT:
- Select 2-4 activities per day from the available activities list.
- The daily_cost must equal the sum of activity costs for that day.
- The total_cost must equal the sum of all daily_costs.
- The total_cost MUST NOT exceed the traveler's budget of {budget}.
- Each activity must come from the available activities list for that specific date.
- Weather descriptions should reflect the actual forecast data provided.
- Respond with ONLY the JSON object. No explanations before or after."""

print("ITINERARY_AGENT_SYSTEM_PROMPT defined.")
print(f"Prompt length: {len(ITINERARY_AGENT_SYSTEM_PROMPT)} characters")

ITINERARY_AGENT_SYSTEM_PROMPT defined.
Prompt length: 2497 characters


### Generate Initial Itinerary

We gather weather and activity data for each day, populate the prompt, and call the LLM to generate the initial travel plan.

In [6]:
def gather_context(vacation: VacationInfo) -> tuple[list[dict], dict[str, list[dict]]]:
    """Gather weather and activity data for all trip dates."""
    start = datetime.strptime(vacation.start_date, "%Y-%m-%d")
    end = datetime.strptime(vacation.end_date, "%Y-%m-%d")
    
    weather_list = []
    activities_dict = {}
    
    current = start
    while current <= end:
        date_str = current.strftime("%Y-%m-%d")
        weather_list.append(get_weather_by_date(vacation.destination_city, date_str))
        activities_dict[date_str] = get_activities_by_date(vacation.destination_city, date_str)
        current += timedelta(days=1)
    
    return weather_list, activities_dict


def generate_initial_itinerary(vacation: VacationInfo) -> TravelPlan:
    """Use the itinerary agent to generate an initial travel plan."""
    weather_list, activities_dict = gather_context(vacation)
    
    # Format the system prompt with context
    formatted_prompt = ITINERARY_AGENT_SYSTEM_PROMPT.format(
        vacation_info=json.dumps(vacation.model_dump(), indent=2),
        weather_data=json.dumps(weather_list, indent=2),
        activities_data=json.dumps(activities_dict, indent=2),
        budget=vacation.budget
    )
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": formatted_prompt},
            {"role": "user", "content": "Please create a detailed travel plan for this trip."}
        ],
        temperature=0.7,
        max_tokens=3000,
        extra_body=_EXTRA_BODY,
    )
    
    content = response.choices[0].message.content.strip()
    
    # Clean potential markdown code fences
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\s*", "", content)
        content = re.sub(r"\s*```$", "", content)
    
    plan_data = extract_json(content)
    travel_plan = TravelPlan(**plan_data)
    
    return travel_plan


# Generate the initial plan
print("Generating initial itinerary with the Itinerary Agent...")
initial_plan = generate_initial_itinerary(vacation_info)

print(f"\nTravel Plan for: {initial_plan.traveler_name}")
print(f"Destination: {initial_plan.destination}")
print(f"Total Cost: ${initial_plan.total_cost:.2f}")
print(f"Budget: ${vacation_info.budget:.2f}")
print(f"\nSummary: {initial_plan.summary}")

for day in initial_plan.days:
    print(f"\n--- {day.date} ({day.weather}) ---")
    for act in day.activities:
        print(f"  [{act.type}] {act.name} @ {act.location} - ${act.cost:.2f} ({act.duration})")
    print(f"  Day Total: ${day.daily_cost:.2f}")

Generating initial itinerary with the Itinerary Agent...



Travel Plan for: Alice Johnson
Destination: San Francisco
Total Cost: $302.00
Budget: $2000.00

Summary: This 4-day San Francisco itinerary for Alice Johnson balances her interests in technology, food, art, and outdoor activities while adhering to a walking-friendly schedule and vegetarian preferences. The plan leverages the sunny weather on June 16 for biking and outdoor exploration, while reserving indoor activities like the Computer History Museum for the drizzly day on June 17. The total cost of $302.00 is well within the $2000.00 budget.

--- 2025-06-15 (64°F (18°C), Partly cloudy with morning fog, 12 mph wind) ---
  [outdoor] Golden Gate Bridge Walk @ Golden Gate Bridge - $0.00 (2 hours)
  [museum] Exploratorium Museum @ Pier 15, Embarcadero - $30.00 (3 hours)
  [food] Ferry Building Marketplace @ Ferry Building - $45.00 (2 hours)
  Day Total: $75.00

--- 2025-06-16 (68°F (20°C), Sunny and clear, 8 mph wind) ---
  [outdoor] Bike Ride Across Golden Gate @ Golden Gate Bridge to Sa

---

## Step 4: Evaluation Prompts

We define evaluation prompts for multi-dimensional assessment of the travel plan:
1. **Weather-Activity Compatibility** - Are outdoor activities scheduled on good weather days?
2. **City Match** - Does the plan match the requested destination?
3. **Date Match** - Do plan dates align with requested dates?
4. **Budget Adherence** - Is the total cost within budget?
5. **Activity Authenticity** - Are activities real/available for the city and dates?

In [7]:
# ── Evaluation System Prompts ──────────────────────────────────────────────

ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """You are an expert travel plan evaluator. Your role is to assess whether the activities in a travel plan are compatible with the weather conditions for each day.

## Your Task
Analyze the travel plan and evaluate weather-activity compatibility for each day.

## Evaluation Criteria
For each day, check:
- Are outdoor activities (hiking, walking tours, bike rides, kayaking) scheduled on days with good weather (sunny, clear, partly cloudy)?
- Are indoor activities (museums, food tours, libraries) scheduled on days with bad weather (rain, drizzle, storms)?
- Would the planned activities be enjoyable given the temperature and conditions?

## Scoring
- PASS: Activities are well-matched to weather conditions
- PARTIAL: Some activities could be better matched to weather
- FAIL: Activities are poorly matched (e.g., outdoor hiking in heavy rain)

## Output Format
Respond with ONLY a valid JSON object:
{{
    "evaluation": "weather_activity_compatibility",
    "overall_score": "PASS" | "PARTIAL" | "FAIL",
    "days": [
        {{
            "date": "YYYY-MM-DD",
            "score": "PASS" | "PARTIAL" | "FAIL",
            "reasoning": "explanation"
        }}
    ],
    "suggestions": ["list of improvement suggestions if any"]
}}

## Examples
- A "Golden Gate Bridge Walk" on a "Sunny and clear" day → PASS
- A "Bike Ride Across Golden Gate" on an "Overcast with light drizzle" day → FAIL
- A "Computer History Museum" visit on a rainy day → PASS (indoor activity)"""


MULTI_DIMENSIONAL_EVAL_SYSTEM_PROMPT = """You are a comprehensive travel plan evaluator. Assess the travel plan across multiple quality dimensions.

## Travel Plan to Evaluate
{travel_plan}

## Original Vacation Request
{vacation_info}

## Available Activities Database
{activities_data}

## Evaluation Dimensions

### 1. City Match
Does the plan's destination match the requested city?

### 2. Date Match  
Do the plan dates exactly match the requested start_date through end_date?

### 3. Budget Adherence
Is the total_cost within the requested budget? Are daily costs calculated correctly?

### 4. Activity Authenticity
Are the planned activities drawn from the available activities database for each specific date? Are costs and details accurate?

### 5. Weather Compatibility
Are activities appropriate for the weather conditions? (Use the weather-activity compatibility evaluation.)

## Output Format
Respond with ONLY a valid JSON object:
{{
    "evaluations": {{
        "city_match": {{
            "score": "PASS" | "FAIL",
            "details": "explanation"
        }},
        "date_match": {{
            "score": "PASS" | "FAIL",
            "details": "explanation"
        }},
        "budget_adherence": {{
            "score": "PASS" | "FAIL",
            "details": "explanation with numbers"
        }},
        "activity_authenticity": {{
            "score": "PASS" | "PARTIAL" | "FAIL",
            "details": "explanation"
        }},
        "weather_compatibility": {{
            "score": "PASS" | "PARTIAL" | "FAIL",
            "details": "explanation"
        }}
    }},
    "overall_score": "PASS" | "NEEDS_REVISION" | "FAIL",
    "revision_suggestions": ["list of specific improvements needed"]
}}"""

print("Evaluation prompts defined:")
print("  - ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT")
print("  - MULTI_DIMENSIONAL_EVAL_SYSTEM_PROMPT")

Evaluation prompts defined:
  - ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT
  - MULTI_DIMENSIONAL_EVAL_SYSTEM_PROMPT


In [8]:
def run_weather_compatibility_eval(travel_plan: TravelPlan) -> dict:
    """Evaluate weather-activity compatibility using the LLM."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT},
            {"role": "user", "content": f"Evaluate this travel plan:\n{json.dumps(travel_plan.model_dump(), indent=2)}"}
        ],
        temperature=0.3,
        max_tokens=2000,
        extra_body=_EXTRA_BODY,
    )

    content = response.choices[0].message.content.strip()
    return extract_json(content)


def run_multi_dimensional_eval(travel_plan: TravelPlan, vacation: VacationInfo) -> dict:
    """Run comprehensive multi-dimensional evaluation of the travel plan."""
    _, activities_dict = gather_context(vacation)

    formatted_prompt = MULTI_DIMENSIONAL_EVAL_SYSTEM_PROMPT.format(
        travel_plan=json.dumps(travel_plan.model_dump(), indent=2),
        vacation_info=json.dumps(vacation.model_dump(), indent=2),
        activities_data=json.dumps(activities_dict, indent=2)
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": formatted_prompt},
            {"role": "user", "content": "Please evaluate this travel plan across all dimensions."}
        ],
        temperature=0.3,
        max_tokens=2000,
        extra_body=_EXTRA_BODY,
    )

    content = response.choices[0].message.content.strip()
    return extract_json(content)


def run_all_evaluations(travel_plan: TravelPlan, vacation: VacationInfo) -> dict:
    """Run all evaluations and return combined results."""
    print("Running weather-activity compatibility evaluation...")
    weather_eval = run_weather_compatibility_eval(travel_plan)

    print("Running multi-dimensional evaluation...")
    multi_eval = run_multi_dimensional_eval(travel_plan, vacation)

    combined = {
        "weather_compatibility_eval": weather_eval,
        "multi_dimensional_eval": multi_eval,
        "overall_verdict": multi_eval.get("overall_score", "UNKNOWN")
    }

    return combined


# Run evaluations on the initial plan
print("=" * 60)
print("EVALUATING INITIAL PLAN")
print("=" * 60)
eval_results = run_all_evaluations(initial_plan, vacation_info)

print(f"\n{'=' * 60}")
print(f"Overall Verdict: {eval_results['overall_verdict']}")
print(f"{'=' * 60}")

multi = eval_results["multi_dimensional_eval"]
if "evaluations" in multi:
    for dim, result in multi["evaluations"].items():
        score = result.get("score", "N/A")
        print(f"  {dim}: {score}")

if "revision_suggestions" in multi and multi["revision_suggestions"]:
    print(f"\nRevision Suggestions:")
    for s in multi["revision_suggestions"]:
        print(f"  - {s}")


EVALUATING INITIAL PLAN
Running weather-activity compatibility evaluation...


Running multi-dimensional evaluation...



Overall Verdict: PASS
  city_match: PASS
  date_match: PASS
  budget_adherence: PASS
  activity_authenticity: PASS
  weather_compatibility: PASS


---

## Step 5: Tool Definitions

We define four tools that the ReACT revision agent can use to iteratively improve the travel plan:
1. **calculator_tool** - Evaluate math expressions for cost calculations
2. **get_activities_by_date_tool** - Look up available activities for a city/date
3. **run_evals_tool** - Run all evaluations on a travel plan
4. **final_answer_tool** - Submit the final revised plan (exits the ReACT loop)

In [9]:
# ── Tool Definitions ───────────────────────────────────────────────────────

def calculator_tool(expression: str) -> str:
    """Evaluate a mathematical expression and return the result as a string.
    
    Use this tool to calculate costs, sum up daily expenses, or verify budget totals.
    
    Args:
        expression (str): A mathematical expression to evaluate, e.g. "30 + 45 + 42" or "152.0 * 4"
    
    Returns:
        str: The result of the calculation as a string
    """
    try:
        # Only allow safe math operations
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return f"Error: Invalid characters in expression. Only numbers and +-*/.() are allowed."
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error calculating '{expression}': {str(e)}"


def get_activities_by_date_tool(city: str, date: str) -> str:
    """Look up available activities for a specific city and date.
    
    Use this tool to find what activities are available on a given date
    so you can select appropriate ones for the travel plan.
    
    Args:
        city (str): The city name, e.g. "San Francisco"
        date (str): The date in YYYY-MM-DD format, e.g. "2025-06-15"
    
    Returns:
        str: JSON string containing a list of available activities,
             each with name, type, cost, duration, and location fields
    """
    activities = get_activities_by_date(city, date)
    return json.dumps(activities, indent=2)


def run_evals_tool(travel_plan: str) -> str:
    """Run all evaluations (weather compatibility + multi-dimensional) on a travel plan.
    
    This tool must be called BEFORE final_answer_tool to verify the plan quality.
    It checks: city match, date match, budget adherence, activity authenticity,
    and weather-activity compatibility.
    
    Args:
        travel_plan (str): A JSON string representing the complete TravelPlan object
    
    Returns:
        str: JSON string with evaluation results including scores and suggestions
    """
    try:
        plan_data = json.loads(travel_plan)
        plan = TravelPlan(**plan_data)
        results = run_all_evaluations(plan, vacation_info)
        return json.dumps(results, indent=2)
    except json.JSONDecodeError as e:
        return f"Error: Invalid JSON in travel plan: {str(e)}"
    except Exception as e:
        return f"Error running evaluations: {str(e)}"


def final_answer_tool(travel_plan: str) -> str:
    """Submit the final revised travel plan. This tool exits the ReACT loop.
    
    IMPORTANT: Only call this AFTER run_evals_tool has been called and the plan
    has passed evaluation. This tool signals that the revision process is complete.
    
    Args:
        travel_plan (str): A JSON string representing the final TravelPlan object
    
    Returns:
        str: Confirmation message with the validated plan
    """
    try:
        plan_data = json.loads(travel_plan)
        plan = TravelPlan(**plan_data)
        return json.dumps({
            "status": "FINAL_ANSWER_SUBMITTED",
            "plan": plan.model_dump()
        }, indent=2)
    except Exception as e:
        return f"Error: Could not validate final plan: {str(e)}"


# Registry for tool dispatch
TOOLS = {
    "calculator_tool": calculator_tool,
    "get_activities_by_date_tool": get_activities_by_date_tool,
    "run_evals_tool": run_evals_tool,
    "final_answer_tool": final_answer_tool,
}

# Test the tools
print("=== Tool Tests ===")
print(f"calculator_tool('30 + 45 + 42') = {calculator_tool('30 + 45 + 42')}")
print(f"\nget_activities_by_date_tool('San Francisco', '2025-06-15'):")
print(get_activities_by_date_tool("San Francisco", "2025-06-15")[:200] + "...")

=== Tool Tests ===
calculator_tool('30 + 45 + 42') = 117

get_activities_by_date_tool('San Francisco', '2025-06-15'):
[
  {
    "name": "Golden Gate Bridge Walk",
    "type": "outdoor",
    "cost": 0.0,
    "duration": "2 hours",
    "location": "Golden Gate Bridge"
  },
  {
    "name": "Exploratorium Museum",
    "t...


---

## Step 6: ReACT Revision Agent

The revision agent uses the **THINK-ACT-OBSERVE** (ReACT) pattern to iteratively improve the travel plan. It:
1. **THINKs** about what needs to be improved
2. **ACTs** by calling a tool (calculator, activity lookup, evaluation, or final answer)
3. **OBSERVEs** the tool's output and decides the next step

The agent MUST run `run_evals_tool` before calling `final_answer_tool` to ensure quality.

In [10]:
# ── Itinerary Revision Agent System Prompt ─────────────────────────────────

ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = """You are an expert travel plan revision agent. Your role is to review, evaluate, and improve a travel plan using the THINK-ACT-OBSERVE (ReACT) reasoning cycle.

## Your Task
You are given an initial travel plan that may have issues. Your job is to:
1. Review the plan for problems (budget issues, weather incompatibility, missing activities, incorrect costs)
2. Use available tools to gather information and verify the plan
3. Make improvements to create an optimal travel plan
4. Evaluate the revised plan to confirm quality
5. Submit the final plan

## THINK-ACT-OBSERVE (ReACT) Cycle

You MUST follow this reasoning pattern in every response:

### THOUGHT
First, think about the current state:
- What issues have you identified in the plan?
- What information do you need?
- What is the best next action?

Write your thinking after "THOUGHT:" on its own line.

### ACTION
Then, specify exactly ONE tool to call. Write your action as a JSON object after "ACTION:" on its own line.

Format:
ACTION: {{"tool_name": "<name>", "arguments": {{"arg1": "value1"}}}}

### After receiving the OBSERVATION (tool result), repeat the cycle.

## Available Tools

### 1. calculator_tool
- Purpose: Evaluate mathematical expressions for cost calculations
- Parameters: {{"expression": "string"}} — a math expression like "30 + 45 + 42"
- Example: ACTION: {{"tool_name": "calculator_tool", "arguments": {{"expression": "30 + 45 + 42"}}}}

### 2. get_activities_by_date_tool
- Purpose: Look up available activities for a specific city and date
- Parameters: {{"city": "string", "date": "string"}} — city name and YYYY-MM-DD date
- Example: ACTION: {{"tool_name": "get_activities_by_date_tool", "arguments": {{"city": "San Francisco", "date": "2025-06-15"}}}}

### 3. run_evals_tool
- Purpose: Run comprehensive evaluations on a travel plan (weather compatibility, budget, dates, activities)
- Parameters: {{"travel_plan": "string"}} — JSON string of the complete TravelPlan
- Example: ACTION: {{"tool_name": "run_evals_tool", "arguments": {{"travel_plan": "<json>"}}}}

### 4. final_answer_tool
- Purpose: Submit the final revised travel plan and EXIT the revision loop
- Parameters: {{"travel_plan": "string"}} — JSON string of the final TravelPlan
- Example: ACTION: {{"tool_name": "final_answer_tool", "arguments": {{"travel_plan": "<json>"}}}}

## CRITICAL RULES
1. You MUST call `run_evals_tool` BEFORE calling `final_answer_tool` to verify the plan.
2. If evaluations show issues, revise the plan and run evaluations again.
3. Calling `final_answer_tool` exits the loop — only call it when the plan is verified.
4. Always include both THOUGHT and ACTION in every response.
5. Use `calculator_tool` to verify cost calculations.
6. Use `get_activities_by_date_tool` to find replacement activities when needed.
7. Ensure all activities are appropriate for the weather conditions on each day.

## Weather Context
{weather_data}

## Available Activities Context
{activities_data}

## Original Vacation Request
{vacation_info}"""

print("ITINERARY_REVISION_AGENT_SYSTEM_PROMPT defined.")
print(f"Prompt length: {len(ITINERARY_REVISION_AGENT_SYSTEM_PROMPT)} characters")


ITINERARY_REVISION_AGENT_SYSTEM_PROMPT defined.
Prompt length: 3010 characters


### ReACT Loop Implementation

The core agent loop parses THOUGHT and ACTION from LLM responses, executes the requested tool, appends the OBSERVATION, and repeats until `final_answer_tool` is called or max iterations are reached.

In [11]:
def parse_action(content: str) -> Optional[dict]:
    """Parse the ACTION JSON from the agent's response."""
    # Look for ACTION: followed by a JSON object
    action_match = re.search(r'ACTION:\s*(\{.*?\})\s*$', content, re.DOTALL | re.MULTILINE)
    if action_match:
        try:
            return json.loads(action_match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Fallback: look for any JSON with tool_name
    json_match = re.search(r'\{[^{}]*"tool_name"[^{}]*\}', content)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except json.JSONDecodeError:
            pass
    
    return None


def execute_tool(action: dict) -> str:
    """Execute a tool based on the parsed action."""
    tool_name = action.get("tool_name", "")
    arguments = action.get("arguments", {})
    
    if tool_name not in TOOLS:
        return f"Error: Unknown tool '{tool_name}'. Available tools: {list(TOOLS.keys())}"
    
    tool_fn = TOOLS[tool_name]
    
    try:
        return tool_fn(**arguments)
    except TypeError as e:
        return f"Error calling {tool_name}: {str(e)}"


def run_revision_agent(
    system_prompt: str,
    user_message: str,
    tools: dict,
    max_iterations: int = 10
) -> Optional[TravelPlan]:
    """Run the ReACT revision agent loop.
    
    Args:
        system_prompt: The formatted system prompt for the revision agent
        user_message: The initial user message with the plan to revise
        tools: Dictionary of available tool functions
        max_iterations: Maximum number of THINK-ACT-OBSERVE cycles
    
    Returns:
        The final TravelPlan if successful, None otherwise
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    final_plan = None
    evals_run = False
    
    for iteration in range(max_iterations):
        print(f"\n{'─' * 50}")
        print(f"ITERATION {iteration + 1}/{max_iterations}")
        print(f"{'─' * 50}")
        
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.5,
            max_tokens=4000,
        extra_body=_EXTRA_BODY,
    )
        
        content = response.choices[0].message.content
        if not content:
            print("Agent returned empty response. Stopping.")
            break
        
        # Display THOUGHT
        thought_match = re.search(r'THOUGHT:(.*?)(?=ACTION:|$)', content, re.DOTALL)
        if thought_match:
            thought = thought_match.group(1).strip()
            print(f"\nTHOUGHT: {thought[:300]}{'...' if len(thought) > 300 else ''}")
        
        # Parse and execute ACTION
        action = parse_action(content)
        if action is None:
            print("Could not parse ACTION from response. Adding guidance and retrying...")
            messages.append({"role": "assistant", "content": content})
            messages.append({
                "role": "user",
                "content": "Please respond with a valid ACTION in the format: ACTION: {\"tool_name\": \"...\", \"arguments\": {...}}"
            })
            continue
        
        tool_name = action.get("tool_name", "unknown")
        print(f"\nACTION: {tool_name}")
        
        # Add assistant message to history
        messages.append({"role": "assistant", "content": content})
        
        # Execute the tool
        observation = execute_tool(action)
        
        # Track if evals were run
        if tool_name == "run_evals_tool":
            evals_run = True
            print(f"OBSERVATION: (evaluation results received, length={len(observation)})")
        elif tool_name == "final_answer_tool":
            print("OBSERVATION: Final answer submitted!")
            try:
                result = json.loads(observation)
                if result.get("status") == "FINAL_ANSWER_SUBMITTED":
                    final_plan = TravelPlan(**result["plan"])
                    print(f"\nAgent completed in {iteration + 1} iterations.")
                    if not evals_run:
                        print("WARNING: Agent submitted final answer without running evaluations!")
                    break
            except Exception as e:
                print(f"Error parsing final answer: {e}")
                observation = f"Error: {str(e)}. Please try again with valid JSON."
        else:
            obs_preview = observation[:200] + "..." if len(observation) > 200 else observation
            print(f"OBSERVATION: {obs_preview}")
        
        # Add observation to messages
        messages.append({
            "role": "user",
            "content": f"OBSERVATION: {observation}"
        })
    
    if final_plan is None:
        print(f"\nAgent did not produce a final plan within {max_iterations} iterations.")
    
    return final_plan

print("run_revision_agent() defined.")

run_revision_agent() defined.


### Run the Revision Agent

We feed the initial plan to the revision agent and let it iteratively improve the plan through the ReACT loop.

In [12]:
# Prepare context for the revision agent
weather_list, activities_dict = gather_context(vacation_info)

formatted_revision_prompt = ITINERARY_REVISION_AGENT_SYSTEM_PROMPT.format(
    weather_data=json.dumps(weather_list, indent=2),
    activities_data=json.dumps(activities_dict, indent=2),
    vacation_info=json.dumps(vacation_info.model_dump(), indent=2)
)

user_message = f"""Here is the initial travel plan that needs to be reviewed and potentially revised:

{json.dumps(initial_plan.model_dump(), indent=2)}

Please review this plan using the THINK-ACT-OBSERVE cycle. Check for:
1. Weather-activity compatibility (are outdoor activities on good weather days?)
2. Budget adherence (total cost within ${vacation_info.budget})
3. Correct cost calculations (daily_cost = sum of activities, total_cost = sum of daily_costs)
4. Activity authenticity (are activities from the available database?)

Remember: You MUST call run_evals_tool BEFORE calling final_answer_tool."""

# Run the revision agent
print("=" * 60)
print("STARTING REVISION AGENT (ReACT Loop)")
print("=" * 60)

revised_plan = run_revision_agent(
    system_prompt=formatted_revision_prompt,
    user_message=user_message,
    tools=TOOLS,
    max_iterations=10
)

if revised_plan:
    print(f"\n{'=' * 60}")
    print("REVISED PLAN")
    print(f"{'=' * 60}")
    print(f"Traveler: {revised_plan.traveler_name}")
    print(f"Destination: {revised_plan.destination}")
    print(f"Total Cost: ${revised_plan.total_cost:.2f} (Budget: ${vacation_info.budget:.2f})")
    print(f"Summary: {revised_plan.summary}")
    for day in revised_plan.days:
        print(f"\n--- {day.date} ({day.weather}) ---")
        for act in day.activities:
            print(f"  [{act.type}] {act.name} @ {act.location} - ${act.cost:.2f} ({act.duration})")
        print(f"  Day Total: ${day.daily_cost:.2f}")

STARTING REVISION AGENT (ReACT Loop)

──────────────────────────────────────────────────
ITERATION 1/10
──────────────────────────────────────────────────



THOUGHT: I need to review the initial travel plan for Alice Johnson. Let me first run the comprehensive evaluation tool to check for any issues with weather compatibility, budget adherence, cost calculations, and activity authenticity. This will give me a baseline understanding of what needs to be improved.

ACTION: run_evals_tool
Running weather-activity compatibility evaluation...


Running multi-dimensional evaluation...


OBSERVATION: (evaluation results received, length=2541)

──────────────────────────────────────────────────
ITERATION 2/10
──────────────────────────────────────────────────



THOUGHT: The evaluations show that the plan passes all criteria - weather compatibility, budget adherence, activity authenticity, and cost calculations are all correct. However, I should verify the cost calculations manually using the calculator tool to ensure the daily costs and total cost are accurate befo...

ACTION: calculator_tool
OBSERVATION: 75

──────────────────────────────────────────────────
ITERATION 3/10
──────────────────────────────────────────────────



THOUGHT: Day 1 cost calculation is correct (0 + 30 + 45 = 75). Let me verify Day 2's cost calculation.

ACTION: calculator_tool
OBSERVATION: 125

──────────────────────────────────────────────────
ITERATION 4/10
──────────────────────────────────────────────────



THOUGHT: Day 2 cost calculation is correct (35 + 65 + 25 = 125). Let me verify Day 3's cost calculation.

ACTION: calculator_tool
OBSERVATION: 52

──────────────────────────────────────────────────
ITERATION 5/10
──────────────────────────────────────────────────



THOUGHT: Day 3 cost calculation is correct (22 + 30 + 0 = 52). Let me verify Day 4's cost calculation.

ACTION: calculator_tool
OBSERVATION: 50

──────────────────────────────────────────────────
ITERATION 6/10
──────────────────────────────────────────────────



THOUGHT: Day 4 cost calculation is correct (0 + 25 + 25 + 0 = 50). Now let me verify the total cost by summing all daily costs.

ACTION: calculator_tool
OBSERVATION: 302

──────────────────────────────────────────────────
ITERATION 7/10
──────────────────────────────────────────────────



THOUGHT: All cost calculations are verified as correct:
- Day 1: $75 (correct)
- Day 2: $125 (correct)
- Day 3: $52 (correct)
- Day 4: $50 (correct)
- Total: $302 (correct)

The comprehensive evaluations also confirmed:
1. Weather compatibility: PASS - outdoor activities are scheduled on appropriate weather ...

ACTION: final_answer_tool
OBSERVATION: Final answer submitted!

Agent completed in 7 iterations.

REVISED PLAN
Traveler: Alice Johnson
Destination: San Francisco
Total Cost: $302.00 (Budget: $2000.00)
Summary: This 4-day San Francisco itinerary for Alice Johnson balances her interests in technology, food, art, and outdoor activities while adhering to a walking-friendly schedule and vegetarian preferences. The plan leverages the sunny weather on June 16 for biking and outdoor exploration, while reserving indoor activities like the Computer History Museum for the drizzly day on June 17. The total cost of $302.00 is well within the $2000.00 budget.

--- 2025-06-15 (64°F (18°C), P

---

## Step 7: Narrative Summary

Generate a story-like narrative recap of the trip using the final revised plan. This transforms the structured data into an engaging, readable trip description.

In [13]:
NARRATIVE_SYSTEM_PROMPT = """You are a creative travel writer. Given a structured travel plan, write an engaging, story-like narrative summary of the trip.

Write in second person ("you") to make it immersive. Include:
- A vivid opening that sets the scene
- Day-by-day highlights woven into a flowing narrative (not just a list)
- Sensory details about locations, food, and experiences
- Weather descriptions that enhance the storytelling
- A satisfying conclusion

Keep it to 3-5 paragraphs. Make the reader excited about this trip!"""


def generate_narrative_summary(plan: TravelPlan) -> str:
    """Generate a story-like narrative summary of the travel plan."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": NARRATIVE_SYSTEM_PROMPT},
            {"role": "user", "content": f"Write a narrative summary for this trip:\n{json.dumps(plan.model_dump(), indent=2)}"}
        ],
        temperature=0.8,
        max_tokens=1500,
        extra_body=_EXTRA_BODY,
    )
    return response.choices[0].message.content


# Use the revised plan if available, otherwise fall back to initial plan
final_plan = revised_plan if revised_plan else initial_plan

print("Generating narrative summary...\n")
narrative = generate_narrative_summary(final_plan)

display(Markdown("---"))
display(Markdown("## Your San Francisco Adventure"))
display(Markdown(narrative))
display(Markdown("---"))


Generating narrative summary...



---

## Your San Francisco Adventure

Your adventure begins on June 15th as you step into the cool, crisp embrace of San Francisco, where the morning fog clings to the iconic Golden Gate Bridge. You spend your first hours walking the span of this engineering marvel, the wind whipping through your hair at a brisk 12 miles per hour while the bay sparkles below. Afterward, you dive into the interactive wonders of the Exploratorium at Pier 15, engaging your mind before satisfying your appetite at the Ferry Building Marketplace. There, amidst the bustle of vendors, you sample artisanal treats, the salty air mingling with the scent of fresh baked bread, marking the perfect end to a day that cost you a mere fraction of your budget.

The sun greets you on June 16th, blessing you with a clear, 68-degree day that is practically made for exploration. You seize the opportunity to bike across the Golden Gate Bridge, pedaling hard into the headwinds until you reach the charming seaside town of Sausalito. The journey works up an appetite for a vibrant food tour through the Mission District, where the savory spices of authentic local cuisine dance on your palate. You cap off the sunshine-filled afternoon soaking in modern masterpieces at the SFMOMA, surrounded by the creative energy of the SoMa district, having experienced the city's artistic and culinary heart in full swing.

By the 17th, the weather shifts to moody elegance with a light drizzle and overcast skies, inviting you indoors. You take a day trip to Mountain View to wander through the Computer History Museum, marveling at the machines that changed the world, safe from the rain. Back in the city, the gloom lifts just enough for a sweet indulgence at Ghirardelli Square, where rich, melted chocolate warms you against the chill. You find a quiet moment of architectural beauty at the Public Library and Civic Center, appreciating the city's culture as the rain patters softly against the grand stonework.

Your final day on June 18th brings back the breeze and the sun for a sweeping farewell. You start with a postcard-perfect view of the Painted Ladies at Alamo Square, the famous Victorian houses glowing against the blue sky. A quick stop at Tartine Bakery offers the world-famous morning bun, its flaky, cinnamon-sugar crust melting in your mouth, before you lose yourself in the psychedelic history of Haight-Ashbury. As evening approaches, you hike up Twin Peaks, the city sprawling out beneath you in a glittering panorama. Watching the sunset paint the sky in hues of violet and gold, you realize you’ve tasted, seen, and felt the true soul of San Francisco, all while staying comfortably under budget.

---

---

## Final Plan Export

Export the final travel plan as a formatted JSON for reference.

In [14]:
# Display the final structured plan
print("=" * 60)
print("FINAL TRAVEL PLAN (Structured JSON)")
print("=" * 60)
print(json.dumps(final_plan.model_dump(), indent=2))

print(f"\n{'=' * 60}")
print(f"Trip for: {final_plan.traveler_name}")
print(f"Destination: {final_plan.destination}")
print(f"Dates: {final_plan.days[0].date} to {final_plan.days[-1].date}")
print(f"Total Cost: ${final_plan.total_cost:.2f}")
print(f"Budget Remaining: ${vacation_info.budget - final_plan.total_cost:.2f}")
print(f"Number of Activities: {sum(len(d.activities) for d in final_plan.days)}")
print(f"{'=' * 60}")

FINAL TRAVEL PLAN (Structured JSON)
{
  "traveler_name": "Alice Johnson",
  "destination": "San Francisco",
  "days": [
    {
      "date": "2025-06-15",
      "weather": "64\u00b0F (18\u00b0C), Partly cloudy with morning fog, 12 mph wind",
      "activities": [
        {
          "name": "Golden Gate Bridge Walk",
          "type": "outdoor",
          "cost": 0.0,
          "duration": "2 hours",
          "location": "Golden Gate Bridge"
        },
        {
          "name": "Exploratorium Museum",
          "type": "museum",
          "cost": 30.0,
          "duration": "3 hours",
          "location": "Pier 15, Embarcadero"
        },
        {
          "name": "Ferry Building Marketplace",
          "type": "food",
          "cost": 45.0,
          "duration": "2 hours",
          "location": "Ferry Building"
        }
      ],
      "daily_cost": 75.0
    },
    {
      "date": "2025-06-16",
      "weather": "68\u00b0F (20\u00b0C), Sunny and clear, 8 mph wind",
      "activit